# Практическая работа №8. Эталонные решения и ответы (Вариант №01)
## Методический ориентир для проверки работ (Easy, Medium, Hard)

---
## 1. Уровень 01 — Easy (5 таблиц, Вариант №01)

### 1.1. Эталонный DDL-скрипт PostgreSQL (5 таблиц)
```sql
DROP TABLE IF EXISTS bar_orders CASCADE;
DROP TABLE IF EXISTS sessions CASCADE;
DROP TABLE IF EXISTS pcs CASCADE;
DROP TABLE IF EXISTS members CASCADE;
DROP TABLE IF EXISTS tariffs CASCADE;

CREATE TABLE tariffs (
    tariff_id   SERIAL        PRIMARY KEY,
    zone_name   VARCHAR(20)   NOT NULL UNIQUE,
    hourly_rate NUMERIC(8, 2) NOT NULL CHECK (hourly_rate > 0),
    is_vip      BOOLEAN       NOT NULL DEFAULT FALSE
);

CREATE TABLE members (
    member_id   SERIAL         PRIMARY KEY,
    nickname    VARCHAR(30)    NOT NULL UNIQUE,
    balance     NUMERIC(10, 2) NOT NULL DEFAULT 0.00 CHECK (balance >= 0),
    is_verified BOOLEAN        NOT NULL DEFAULT FALSE,
    reg_date    DATE           NOT NULL DEFAULT CURRENT_DATE
);

CREATE TABLE pcs (
    pc_id     SERIAL      PRIMARY KEY,
    pc_code   VARCHAR(10) NOT NULL UNIQUE,
    tariff_id INTEGER     NOT NULL REFERENCES tariffs(tariff_id),
    ram_gb    SMALLINT    NOT NULL CHECK (ram_gb IN (16, 32, 64)),
    is_active BOOLEAN     NOT NULL DEFAULT TRUE
);

CREATE TABLE sessions (
    session_id   SERIAL   PRIMARY KEY,
    member_id    INTEGER  REFERENCES members(member_id),
    pc_id        INTEGER  NOT NULL REFERENCES pcs(pc_id),
    hours        SMALLINT NOT NULL CHECK (hours BETWEEN 1 AND 24),
    session_date DATE     NOT NULL DEFAULT CURRENT_DATE
);

CREATE TABLE bar_orders (
    order_id     SERIAL        PRIMARY KEY,
    session_id   INTEGER       NOT NULL REFERENCES sessions(session_id),
    item_name    VARCHAR(40)   NOT NULL,
    price        NUMERIC(8, 2) NOT NULL CHECK (price > 0),
    is_delivered BOOLEAN       NOT NULL DEFAULT FALSE
);
```

### 1.2. Ответы к запросам уровня Easy (Вариант №01)

In [1]:
import os
import sqlite3
import pandas as pd

conn_easy = sqlite3.connect(os.path.join("easy", "club_easy.db"))

print("Запрос 1 (Easy): WHERE zone_name = 'VIP'")
display(pd.read_sql("SELECT * FROM tariffs WHERE zone_name = 'VIP';", conn_easy))

print("Запрос 2 (Easy): WHERE item_name ILIKE '%cola%'")
display(pd.read_sql("SELECT * FROM bar_orders WHERE LOWER(item_name) LIKE '%cola%';", conn_easy))

print("Запрос 3 (Easy): WHERE is_active = TRUE AND ram_gb >= 32")
display(pd.read_sql("SELECT pc_id, pc_code, ram_gb, is_active FROM pcs WHERE is_active = 1 AND ram_gb >= 32 ORDER BY ram_gb DESC;", conn_easy))

conn_easy.close()

Запрос 1 (Easy): WHERE zone_name = 'VIP'
Запрос 2 (Easy): WHERE item_name ILIKE '%cola%'
Запрос 3 (Easy): WHERE is_active = TRUE AND ram_gb >= 32


tariff_id,zone_name,hourly_rate,is_vip
3,VIP,380.0,1


order_id,session_id,item_name,price,is_delivered
12,12,Coca-Cola Zero 0.5,280.0,1


pc_id,pc_code,ram_gb,is_active
2,VIP-02,32,1
4,VIP-04,32,1
6,VIP-06,32,1
7,BOOT-07,32,1
9,BOOT-09,32,1
10,BOOT-10,32,1


---
## 2. Уровень 02 — Medium (6 таблиц, Вариант №01)

### 2.1. Добавляемая 6-я таблица `tournament_entries`
```sql
CREATE TABLE tournament_entries (
    entry_id   SERIAL         PRIMARY KEY,
    member_id  INTEGER        NOT NULL REFERENCES members(member_id),
    pc_id      INTEGER        NOT NULL REFERENCES pcs(pc_id),
    game_title VARCHAR(40)    NOT NULL,
    prize      NUMERIC(10, 2) CONSTRAINT chk_entry_prize CHECK (prize IS NULL OR prize >= 0),
    CONSTRAINT uq_member_game UNIQUE (member_id, game_title)
);
```

### 2.2. Ответы к запросам уровня Medium (Вариант №01)

In [2]:
conn_med = sqlite3.connect(os.path.join("medium", "club_medium.db"))

print("Запрос 1 (Medium): Поиск '%energy%' или '%cola%' через ILIKE в bar_orders")
display(pd.read_sql("SELECT order_id, session_id, item_name, price, is_delivered FROM bar_orders WHERE LOWER(item_name) LIKE '%energy%' OR LOWER(item_name) LIKE '%cola%' ORDER BY price DESC;", conn_med))

print("Запрос 2 (Medium): Замена NULL через COALESCE(prize, 0.00) в tournament_entries")
display(pd.read_sql("SELECT entry_id, member_id, game_title, prize, COALESCE(prize, 0.00) AS prize_clean FROM tournament_entries ORDER BY prize_clean DESC LIMIT 10;", conn_med))

print("Запрос 3 (Medium): Верифицированные игроки по дате регистрации (reg_date)")
display(pd.read_sql("SELECT member_id, nickname, reg_date, balance FROM members WHERE is_verified = 1 ORDER BY reg_date ASC LIMIT 10;", conn_med))

conn_med.close()

Запрос 1 (Medium): Поиск '%energy%' или '%cola%' через ILIKE в bar_orders
Запрос 2 (Medium): Замена NULL через COALESCE(prize, 0.00) в tournament_entries
Запрос 3 (Medium): Верифицированные игроки по дате регистрации (reg_date)


order_id,session_id,item_name,price,is_delivered
36,55,Red Bull Energy 0.25,440.0,1
5,2,Red Bull Energy 0.25,220.0,1
24,6,Red Bull Energy 0.25,220.0,1
35,9,Red Bull Energy 0.25,220.0,1
37,1,Red Bull Energy 0.25,220.0,0
43,29,Red Bull Energy 0.25,220.0,0
13,4,Coca-Cola Zero 0.5,140.0,1
23,3,Coca-Cola Zero 0.5,140.0,1


entry_id,member_id,game_title,prize,prize_clean
4,6,League of Legends,15000.0,15000.0
6,1,Apex Legends,15000.0,15000.0
10,4,Apex Legends,10000.0,10000.0
21,28,Valorant,10000.0,10000.0
14,3,Apex Legends,2500.0,2500.0
17,6,PUBG: Battlegrounds,2500.0,2500.0
20,22,PUBG: Battlegrounds,2500.0,2500.0
23,3,League of Legends,2500.0,2500.0
27,16,PUBG: Battlegrounds,2500.0,2500.0
30,12,Dota 2,2500.0,2500.0


member_id,nickname,reg_date,balance
14,NeonViper_64,2025-09-14,3070.74
20,LunarEclipse_55,2025-09-15,278.35
7,IronClad_98,2025-09-22,0.00
29,NightStalker_62,2025-09-26,2211.62
6,GoldenEagle_50,2025-10-13,2034.33
3,CobaltFalcon_87,2025-10-17,1529.61
5,PhantomAce_47,2025-11-02,1750.73
18,StormBreaker_65,2025-11-05,3493.06
23,PixelHunter_91,2025-11-05,435.35
19,HyperNova_89,2025-11-06,1543.86


---
## 3. Уровень 03 — Hard (6 таблиц, `ON DELETE`, `UPSERT` и `FILTER`, Вариант №01)

### 3.1. Ответы к аналитическим запросам уровня Hard (Вариант №01)

In [3]:
conn_hard = sqlite3.connect(os.path.join("hard", "club_hard.db"))

print("Запрос 2 (Hard): Сводка по таблице pcs с использованием FILTER (WHERE ...)")
q_hard_2 = """
SELECT
    COUNT(*) AS total_pcs,
    COUNT(*) FILTER (WHERE is_active = 1) AS active_pcs,
    COUNT(*) FILTER (WHERE is_active = 0) AS repair_pcs,
    COUNT(*) FILTER (WHERE ram_gb = 32) AS ram_32gb_pcs,
    COUNT(*) FILTER (WHERE ram_gb = 64) AS ram_64gb_pcs
FROM pcs;
"""
display(pd.read_sql(q_hard_2, conn_hard))

print("Запрос 3 (Hard): Сводка по таблице bar_orders с использованием FILTER (WHERE ...)")
q_hard_3 = """
SELECT
    COUNT(*) AS total_orders,
    ROUND(SUM(price), 2) AS total_revenue,
    ROUND(SUM(price) FILTER (WHERE is_delivered = 1), 2) AS delivered_revenue,
    ROUND(SUM(price) FILTER (WHERE LOWER(item_name) LIKE '%cola%' OR LOWER(item_name) LIKE '%red bull%' OR LOWER(item_name) LIKE '%monster%'), 2) AS energy_and_cola_revenue
FROM bar_orders;
"""
display(pd.read_sql(q_hard_3, conn_hard))

print("Запрос 4 (Hard): Статистика по дисциплинам из tournament_entries (для экспорта в hard_tournaments_var01.csv)")
q_hard_4 = """
SELECT
    game_title,
    COUNT(*) AS total_entries,
    COUNT(prize) AS winners_count,
    ROUND(COALESCE(SUM(prize), 0.00), 2) AS total_prize_paid,
    ROUND(AVG(COALESCE(prize, 0.00)), 2) AS avg_prize_per_entry
FROM tournament_entries
GROUP BY game_title
ORDER BY total_prize_paid DESC;
"""
display(pd.read_sql(q_hard_4, conn_hard))

conn_hard.close()

Запрос 2 (Hard): Сводка по таблице pcs с использованием FILTER (WHERE ...)
Запрос 3 (Hard): Сводка по таблице bar_orders с использованием FILTER (WHERE ...)
Запрос 4 (Hard): Статистика по дисциплинам из tournament_entries (для экспорта в hard_tournaments_var01.csv)


total_pcs,active_pcs,repair_pcs,ram_32gb_pcs,ram_64gb_pcs
16,13,3,6,0


total_orders,total_revenue,delivered_revenue,energy_and_cola_revenue
65,18160.0,12230.0,3610.0


game_title,total_entries,winners_count,total_prize_paid,avg_prize_per_entry
Apex Legends,11,7,48500.0,4409.09
PUBG: Battlegrounds,9,6,38500.0,4277.78
League of Legends,8,7,23000.0,2875.00
Dota 2,8,3,13500.0,1687.50
Valorant,4,1,10000.0,2500.00
Counter-Strike 2,5,1,10000.0,2000.00
